In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive to the Colab virtual machine
drive.mount('/content/drive')

# 2. Define your primary workspace path
WORKSPACE = "/content/drive/MyDrive/DevOps-SFT-Generator"

# 3. Create the workspace if it doesn't exist, then step inside
os.makedirs(WORKSPACE, exist_ok=True)
os.chdir(WORKSPACE)

print(f"\nSuccess! Your working directory is now set to: {os.getcwd()}")

In [ ]:
import json
import os

progress_file = os.path.join(WORKSPACE, ".processing_progress.json")

if os.path.exists(progress_file):
    try:
        with open(progress_file, 'r', encoding='utf-8') as f:
            progress_data = json.load(f)

        # Convert any legacy Windows path formats to cross-platform Linux formatting
        normalized_data = {}
        for key, value in progress_data.items():
            normalized_key = key.replace('\\\\', '/').replace('\\', '/')
            normalized_data[normalized_key] = value

        with open(progress_file, 'w', encoding='utf-8') as f:
            json.dump(normalized_data, f, indent=4)
        print("Path normalization complete! Windows paths converted to Linux format.")
    except Exception as e:
        print(f"Progress file found but couldn't be normalized: {e}. Starting fresh.")
else:
    # Initialize a clean tracking map
    with open(progress_file, 'w', encoding='utf-8') as f:
        json.dump({}, f)
    print("Created a brand new .processing_progress.json file on Google Drive.")

In [ ]:
import sys
import os

# Define a lightweight package storage directory on Google Drive
PERSISTENT_LIBS = "/content/drive/MyDrive/DevOps-SFT-Generator/python_libs"
os.makedirs(PERSISTENT_LIBS, exist_ok=True)

if PERSISTENT_LIBS not in sys.path:
    sys.path.insert(0, PERSISTENT_LIBS)

try:
    import ollama
    from tqdm.notebook import tqdm
    print("Lightweight generation libraries loaded cleanly from Google Drive cache!")
except ImportError:
    print("Cache empty. Installing ONLY the required tools (takes less than 5 seconds)...")
    # We install ONLY ollama and tqdm. No heavy AI packages.
    %pip install --target={PERSISTENT_LIBS} ollama tqdm
    print("Done! Please run this cell one more time to complete imports.")

In [ ]:
import os

# Set environment variable mapping model storage to your permanent Drive folder
OLLAMA_DRIVE_DIR = "/content/drive/MyDrive/DevOps-SFT-Generator/ollama_models"
os.makedirs(OLLAMA_DRIVE_DIR, exist_ok=True)
%env OLLAMA_MODELS={OLLAMA_DRIVE_DIR}

print("1. Installing zstd extractor...")
!apt-get update -qq && apt-get install -y -qq zstd

print("\n2. Downloading the official Ollama archive (~1.3 GB, contains GPU drivers)...")
# Download the new .tar.zst packaging format directly from the latest release
!curl -L https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64.tar.zst -o ollama.tar.zst

print("\n3. Extracting engine to system paths...")
# Extract bin/ and lib/ directories directly to /usr
!tar -C /usr -I zstd -xf ollama.tar.zst

print("\n4. Cleaning up temporary files...")
!rm ollama.tar.zst

print("\n5. Verification Check:")
!ollama --version

In [ ]:
import os
import time
import requests
import subprocess

print("Clearing out ghost processes...")
os.system("pkill ollama")
time.sleep(2)

print("Booting fresh Ollama daemon process in the background...")
# Start server independently so it doesn't block notebook execution
process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Give the server a moment to bind to the port
time.sleep(5)

MODEL_NAME = "qwen2.5-coder:7b"
print(f"Verifying local GDrive model weights cache for '{MODEL_NAME}'...")
os.system(f"ollama pull {MODEL_NAME}")

try:
    res = requests.get("http://127.0.0.1:11434/")
    if res.status_code == 200:
        print(f"\nSUCCESS! System Online! Ollama is ready using {MODEL_NAME}.")
except Exception as e:
    print(f"\nServer is unreachable. Please verify model paths and logs.")

In [ ]:
import os
import sys
import json
import glob
import time
import hashlib
from collections import Counter
import ollama

# Force disable the buggy tqdm monitor background thread globally
os.environ["TQDM_DISABLE_MONITOR"] = "1"
from tqdm.notebook import tqdm

# --- Custom Exception for Clean Jupyter Exit ---
class OllamaConnectionError(Exception):
    """Custom exception to halt cell execution cleanly."""
    pass

# --- CORE SETTINGS ---
WORKSPACE = "/content/drive/MyDrive/DevOps-SFT-Generator"
DATA_DIR = os.path.join(WORKSPACE, "02_clean_data")
OUTPUT_FILE = os.path.join(WORKSPACE, "raft_qa_dataset.jsonl")
PROGRESS_FILE = os.path.join(WORKSPACE, ".processing_progress.json")
MODEL_NAME = "qwen2.5-coder:7b"
CHUNK_SIZE = 3500
TARGET_QA_COUNT = 7000

# --- SYSTEM PROMPT (Enforces Detailed 7-Section SFT Quality Guardrails) ---
SYSTEM_PROMPT = """You are a Senior DevOps Engineer with 10+ years of production experience across cloud infrastructure, CI/CD pipelines, containerization, observability, and platform engineering. You are also an expert SFT dataset curator who understands what makes a training example genuinely useful for fine-tuning LLMs.

Your task is to analyze a technical documentation chunk and generate 2-4 high-quality Question-Answer pairs for supervised fine-tuning.

===========================================
SECTION 1: CATEGORY DEFINITIONS & TRIGGERS
===========================================

Only generate a category if the source text DIRECTLY supports it. Never hallucinate facts.

- text_conceptual
  WHEN: Text explains \"why\" something works, compares tools, or describes system behavior.
  FOCUS: Trade-offs, mental models, architectural decisions, failure modes.
  EXAMPLE TRIGGER: \"Explain when to use Kafka over RabbitMQ for a microservices event pipeline.\"

- text_howto
  WHEN: Text describes a process, workflow, or sequence of administrative steps.
  FOCUS: Ordered prerequisites, exact commands, expected outputs at each step.
  EXAMPLE TRIGGER: \"Walk me through setting up mutual TLS between two Kubernetes services from scratch.\"

- text_troubleshooting
  WHEN: Text mentions errors, failure modes, degraded states, or recovery procedures.
  FOCUS: Exact error message in instruction, root cause identification, resolution steps with verification.
  EXAMPLE TRIGGER: \"My Terraform apply fails with 'Error: Invalid provider configuration' after upgrading to v1.5. How do I fix it?\"

- code_scripting
  WHEN: Text describes a repeatable operational task that benefits from automation.
  FOCUS: Production-grade Bash/Python with argument parsing, error handling, logging, and exit codes.
  EXAMPLE TRIGGER: \"Write a Bash script that checks all EKS node groups for version drift and sends a Slack alert if any node is 2+ minor versions behind the control plane.\"

- code_iac
  WHEN: Text describes infrastructure, configuration, or deployment topology.
  FOCUS: Complete, deployable IaC files — no partial configs. Include variable declarations, outputs, and inline comments.
  EXAMPLE TRIGGER: \"Write a Terraform module that provisions an RDS PostgreSQL instance with encrypted storage, automated backups, and a parameter group enforcing SSL connections.\"

- code_refactoring
  WHEN: Text shows or implies a pattern that has known security, performance, or reliability issues.
  FOCUS: Show the BEFORE (broken/insecure version), explain WHY it's wrong, then show the AFTER with improvements annotated.
  EXAMPLE TRIGGER: \"Refactor this Dockerfile that runs the app as root and copies the entire repo into the image.\"

===================================
SECTION 2: DIFFICULTY LEVEL GUIDE
===================================

Assign one difficulty to each pair based on assumed prior knowledge:

  beginner     -> Junior engineer, knows basic Linux/Git, needs hand-holding on concepts
  intermediate -> Mid-level engineer, comfortable with the tool, needs production-grade patterns
  advanced     -> Senior engineer, needs edge cases, internals, performance tuning, or security hardening

Distribute across difficulty levels — do not generate 3 intermediate pairs from one chunk.

================================
SECTION 3: RESPONSE FORMATTING
================================

FORMAT RULES BY CATEGORY:

text_conceptual responses must follow:
  1. Direct answer in 1-2 sentences
  2. Core reasoning / mechanism (2-4 sentences)
  3. Trade-off table or bullet comparison if comparing tools
  4. When to use / when NOT to use (1-2 sentences each)

text_howto responses must follow:
  1. Prerequisites (tools, permissions, env vars needed)
  2. Numbered steps with exact commands where applicable
  3. Expected output or success indicator per step
  4. Common mistakes / gotchas at the end

text_troubleshooting responses must follow:
  1. Root cause (1-2 sentences — be specific)
  2. Diagnostic commands to confirm the root cause
  3. Step-by-step fix
  4. Verification command to confirm resolution
  5. Prevention tip to avoid recurrence

code_scripting / code_iac / code_refactoring responses must follow:
  1. Brief explanation of what the code does and key design decisions (2-3 sentences)
  2. Complete code block in a fenced block with language tag (```bash, ```python, ```hcl, ```yaml, etc.)
  3. Usage example showing how to invoke/deploy it
  4. For refactoring: explicit BEFORE block, then explanation of each change, then AFTER block

==================================
SECTION 4: INSTRUCTION PHRASING
==================================

Instructions must sound like a real engineer typed them under time pressure. Use varied openers:

  Scenario-based : \"We're migrating from X to Y and hitting [specific problem]. How do we...\"
  Direct task    : \"Write a [specific artifact] that [specific requirement] without [specific constraint].\"
  Diagnostic     : \"Getting '[exact error string]' when running [command]. What's causing this?\"
  Conceptual     : \"Why does [technology] behave differently than expected when [specific condition]?\"
  Comparative    : \"Our team is debating between [A] and [B] for [use case]. What are the real trade-offs?\"
  Operational    : \"How do I [task] in [environment] while ensuring [constraint like zero-downtime]?\"

NEVER use these stale openers: \"Can you explain...\", \"Please provide...\", \"Describe the...\"

==================================
SECTION 5: HARD QUALITY RULES
==================================

DO:
  - Use real tool names, real flag names, real API fields (kubectl, helm, terraform, aws-cli, etc.)
  - Include version numbers when behavior is version-sensitive
  - Make code blocks copy-paste ready with realistic variable names (not FOO, BAR)
  - For K8s YAML: include apiVersion, kind, metadata, and spec — always complete manifests
  - For Terraform: include required_providers block and at least one output
  - For Bash: include #!/usr/bin/env bash, set -euo pipefail, and usage() function

NEVER:
  - Use placeholders: <YOUR_VALUE>, <REPLACE_ME>, INSERT_HERE, ...
  - Truncate responses mid-answer
  - Generate a pair when the source text is too sparse to support an accurate answer
  - Repeat the same instruction phrasing across pairs in one batch
  - Invent command flags or API parameters that don't exist
  - Add markdown outside the JSON output (no ```json fences wrapping the output)

====================================
SECTION 6: SELF-VALIDATION CHECKLIST
====================================

Before finalizing output, mentally verify each pair:

  - Is the instruction phrased the way a real engineer would ask it?
  - Does the response fully answer the instruction with no gaps?
  - Is all code syntactically valid and production-realistic?
  - Does the difficulty level match the complexity of the answer?
  - Is the category correctly assigned (not just closest-guess)?
  - Are there zero placeholders or truncations in the response?
  - Do the pairs in this batch cover different categories and difficulties?

If any box would be unchecked, revise the pair or drop it entirely.

==============================
SECTION 7: OUTPUT JSON SCHEMA
==============================

Output ONLY a raw JSON object. No markdown. No preamble. No explanation outside the JSON.

{
  \"qa_pairs\": [
    {
      \"instruction\": \"Scenario-driven, specific engineer question here\",
      \"response\": \"Complete, structured, production-grade answer here\",
      \"category\": \"code_iac\",
      \"difficulty\": \"intermediate\",
      \"tags\": [\"terraform\", \"rds\", \"aws\", \"encryption\"]
    }
  ]
}

The \"tags\" field must list 2-5 specific technology/tool/concept keywords from the answer — used for dataset filtering downstream.
"""

# --- 1. SMARTER CONTEXT-AWARE CHUNKER ---
def chunk_text(text, max_chars=3500):
    """Context-aware chunker that respects code blocks, headers, and maintains overlap context."""
    chunks = []
    current_chunk = []
    current_length = 0
    in_code_block = False

    lines = text.split('\n')
    for line in lines:
        if line.strip().startswith('```'):
            in_code_block = not in_code_block

        is_header = line.startswith('#') and not in_code_block

        if is_header and current_length > max_chars * 0.5 and current_chunk:
            chunks.append('\n'.join(current_chunk))
            current_chunk = [line]
            current_length = len(line)
        elif current_length + len(line) > max_chars and not in_code_block and current_chunk:
            chunks.append('\n'.join(current_chunk))
            current_chunk = current_chunk[-3:] + [line]
            current_length = sum(len(l) for l in current_chunk)
        else:
            current_chunk.append(line)
            current_length += len(line)

    if current_chunk:
        chunks.append('\n'.join(current_chunk))

    return [c for c in chunks if len(c.strip()) > 100]

# --- 2. VALIDATION ENGINE ---
def validate_qa_pair(pair: dict) -> tuple[bool, str]:
    """Filters out low-quality, broken, or truncated pairs before committing to disk."""
    instruction = pair.get("instruction", "").strip()
    response = pair.get("response", "").strip()
    category = pair.get("category", "").strip()
    difficulty = pair.get("difficulty", "").strip()
    tags = pair.get("tags", [])

    valid_categories = {
        "text_conceptual", "text_howto", "text_troubleshooting",
        "code_scripting", "code_iac", "code_refactoring"
    }

    valid_difficulties = {"beginner", "intermediate", "advanced"}

    if len(instruction) < 20:
        return False, "instruction too short"
    if len(response) < 80:
        return False, "response too short (likely truncated)"
    if category not in valid_categories:
        return False, f"invalid category: {category}"
    if difficulty not in valid_difficulties:
        return False, f"invalid difficulty schema: {difficulty}"
    if not isinstance(tags, list) or len(tags) < 2 or len(tags) > 5:
        return False, f"invalid tags schema: must be list of 2-5 strings, got {tags}"
    if category.startswith("code_") and "```" not in response:
        return False, "code category missing markdown code blocks"

    bad_patterns = ["<YOUR_", "...", "placeholder", "TODO:", "etc."]
    if any(p in response for p in bad_patterns):
        return False, "response contains placeholders or truncation marks"

    if category == "text_troubleshooting":
        error_signals = ["error", "failed", "cannot", "permission denied", "exit code", "crash", "exception"]
        if not any(s in instruction.lower() for s in error_signals):
            return False, "troubleshooting instruction lacks specific error markers"

    return True, "Valid"

# --- 3. CATEGORY-BALANCED SAMPLING HINT ---
def get_category_hint(counts: Counter) -> str:
    all_cats = ["text_conceptual", "text_howto", "text_troubleshooting",
                "code_scripting", "code_iac", "code_refactoring"]
    if not counts:
        return ""
    max_val = counts.most_common(1)[0][1]
    underrepresented = [c for c in all_cats if counts.get(c, 0) < max_val * 0.7]
    if underrepresented:
        return f"\n\nPRIORITY BALANCING HINT: Your output should prioritize generating items under these specific underrepresented categories if the source context supports them: {', '.join(underrepresented)}."
    return ""

# --- 4. STARTUP STATE HYDRATION & AUTO-REPAIR ---
current_qa_count = 0
seen_instructions = set()
category_counts = Counter()
successful_files = set()

if os.path.exists(OUTPUT_FILE):
    print("Hydrating session states from existing dataset...")
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                try:
                    record = json.loads(line)
                    messages = record.get("messages", [])
                    user_content = ""
                    for m in messages:
                        if m.get("role") == "user":
                            user_content = m.get("content", "")
                            break

                    if "\n\nContext:\n" in user_content:
                        instr = user_content.split("\n\nContext:\n")[0]
                    else:
                        instr = user_content

                    instr_norm = instr.lower().strip()
                    h = hashlib.md5(instr_norm.encode()).hexdigest()

                    seen_instructions.add(h)
                    current_qa_count += 1

                    source_file = record.get("source_file")
                    if source_file:
                        successful_files.add(source_file)

                    category = record.get("category")
                    if category:
                        category_counts[category] += 1
                except:
                    pass

# Auto-Repair Engine Setup
if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE, 'r', encoding='utf-8') as f:
        progress = json.load(f)
else:
    progress = {}

repaired_progress = {}
poisoned_count = 0

for rel_path, status in progress.items():
    if status == "completed":
        if rel_path in successful_files:
            repaired_progress[rel_path] = "completed"
        else:
            abs_path = os.path.join(WORKSPACE, rel_path)
            if os.path.exists(abs_path):
                with open(abs_path, 'r', encoding='utf-8', errors='ignore') as f:
                    text_content = f.read().strip()
                if not text_content:
                    repaired_progress[rel_path] = "completed"
                else:
                    poisoned_count += 1
            else:
                pass
    else:
        repaired_progress[rel_path] = status

progress = repaired_progress

if poisoned_count > 0:
    print(f"Auto-Repair Check: Cleaned progress database! Restored {poisoned_count} poisoned files back to the generation queue.")
    with open(PROGRESS_FILE, 'w', encoding='utf-8') as f:
        json.dump(progress, f, indent=4)

print(f"Resuming processing at status: {current_qa_count} / {TARGET_QA_COUNT} pairs.")
print(f"   - Unique Deduplication Hashes Registered: {len(seen_instructions)}")
print(f"   - Current Category Distribution: {dict(category_counts)}")

# --- 5. DISCOVER PENDING FILES ---
all_files = []
for file_path in glob.glob(os.path.join(DATA_DIR, "**", "*"), recursive=True):
    if os.path.isfile(file_path):
        relative_path = os.path.relpath(file_path, WORKSPACE).replace('\\', '/')
        all_files.append((relative_path, file_path))

pending_files = [f for f in all_files if progress.get(f[0]) != "completed"]

# --- 6. EXECUTION RUNTIME LOOP ---
target_reached = False
if current_qa_count < TARGET_QA_COUNT and pending_files:
    print(f"Found {len(pending_files)} files left to process. Launching generation pipeline...\n")
    pbar = tqdm(total=TARGET_QA_COUNT, initial=current_qa_count, desc="Overall Progress")

    for rel_path, abs_path in pending_files:
        if current_qa_count >= TARGET_QA_COUNT:
            break

        print(f"Reading: {rel_path}")

        try:
            with open(abs_path, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read()

            if not content.strip():
                progress[rel_path] = "completed"
                with open(PROGRESS_FILE, 'w', encoding='utf-8') as f:
                    json.dump(progress, f, indent=4)
                continue

            chunks = chunk_text(content, max_chars=CHUNK_SIZE)

            for idx, chunk in enumerate(chunks):
                if current_qa_count >= TARGET_QA_COUNT:
                    target_reached = True
                    break

                print(f"   Processing Chunk {idx+1}/{len(chunks)} via Ollama... ", end="")
                start_time = time.time()

                balance_hint = get_category_hint(category_counts)
                prompt = f"Analyze this technical context and generate the QA pairs:{balance_hint}\n\n### CONTEXT:\n{chunk}"

                try:

                    response = ollama.chat(
                        model=MODEL_NAME,
                        messages=[
                            {"role": "system", "content": SYSTEM_PROMPT},
                            {"role": "user", "content": prompt}
                        ],
                        options={
                            "temperature": 0.4,
                            "top_p": 0.9,
                            "repeat_penalty": 1.15,
                            "num_predict": 2048,
                        },
                        format="json"
                    )

                    raw_content = response['message']['content']
                    data = json.loads(raw_content)
                    added_in_chunk = 0

                    if "qa_pairs" in data and isinstance(data["qa_pairs"], list):
                        with open(OUTPUT_FILE, 'a', encoding='utf-8') as out_f:
                            for pair in data["qa_pairs"]:
                                if current_qa_count >= TARGET_QA_COUNT:
                                    target_reached = True
                                    break

                                is_valid, reason = validate_qa_pair(pair)
                                if not is_valid:
                                    continue

                                instr_norm = pair["instruction"].lower().strip()
                                instr_hash = hashlib.md5(instr_norm.encode()).hexdigest()
                                if instr_hash in seen_instructions:
                                    continue

                                seen_instructions.add(instr_hash)
                                category_counts[pair["category"]] += 1

                                # Convert standard instruction/response to ChatML format with Context appended
                                raft_record = {
                                    "messages": [
                                        {"role": "system", "content": "You are a helpful assistant."},
                                        {"role": "user", "content": f"{pair['instruction']}\n\nContext:\n{chunk}"},
                                        {"role": "assistant", "content": pair["response"]}
                                    ],
                                    "source_file": rel_path,
                                    "category": pair["category"],
                                    "difficulty": pair["difficulty"],
                                    "tags": pair["tags"]
                                }

                                out_f.write(json.dumps(raft_record, ensure_ascii=False) + "\n")
                                current_qa_count += 1
                                added_in_chunk += 1
                                pbar.update(1)

                    elapsed = time.time() - start_time
                    print(f"Done! Created +{added_in_chunk} pairs ({elapsed:.1f}s)")

                except json.JSONDecodeError as jde:
                    print(f"Skipped chunk due to JSON parsing error: {jde}")
                    continue
                except Exception as chunk_err:
                    err_msg = str(chunk_err).lower()
                    connection_indicators = [
                        "connection", "connect", "timeout", "network", "remote",
                        "localhost", "127.0.0.1", "status code 500", "http", "api"
                    ]
                    if any(ci in err_msg for ci in connection_indicators):
                        pbar.close()
                        raise OllamaConnectionError(
                            f"\n\nCRITICAL CONNECTION ERROR: Ollama server became unreachable.\n"
                            f"Detail: {chunk_err}\n"
                            f"ACTION REQUIRED: Run your server wake-up cell (Cell 5), then re-run this cell."
                        ) from None
                    else:
                        print(f"Skipped chunk due to model execution error: {chunk_err}")
                        continue

            # Save file state checkpoint
            if not target_reached:
                progress[rel_path] = "completed"
                with open(PROGRESS_FILE, 'w', encoding='utf-8') as f:
                    json.dump(progress, f, indent=4)

        except OllamaConnectionError as oce:
            print(oce)
            break
        except Exception as file_err:
            print(f"Structural file read failure {rel_path}: {file_err}")
            continue

    try:
        pbar.close()
    except:
        pass

if current_qa_count >= TARGET_QA_COUNT:
    print(f"\nDirect hit! Cleanly compiled exactly {current_qa_count} production-ready SFT/RAFT instances.")
else:
    print(f"\nReached file boundary or paused. Final output generated: {current_qa_count} items.")
print(f"Final Dataset Path: {OUTPUT_FILE}")